In [45]:
import time
import chess
import chess.pgn
import chess.svg
import ipywidgets as widgets
from IPython.display import display, clear_output, SVG


class GraphicalGameManager:
    """
    Jupyter Notebook / Google Colab Graphical Game Manager.
    Renders live SVG board updates and interactive UI controls using ipywidgets.
    """
    def __init__(self,
                 white_player,
                 black_player,
                 turn_delay=0.5,
                 board_size=400,
                 pgn_filename ="match"):
        self.board = chess.Board()
        self.white_player = white_player
        self.black_player = black_player
        self.turn_delay = turn_delay
        self.board_size = board_size
        self.is_paused = False
        self.pgn_fllename = pgn_filename

        # --- Setup IPyWidgets UI Layout ---
        self.board_output = widgets.Output()
        self.status_label = widgets.Label(value="Status: Ready to start")
        self.move_log = widgets.Textarea(
            value="",
            description="Moves:",
            disabled=True,
            layout=widgets.Layout(width="250px", height="380px")
        )
        
        # Interactive Buttons
        self.play_button = widgets.Button(description="Start Game", icon="play", button_style="success")
        self.pause_button = widgets.Button(description="Pause", icon="pause", button_style="warning")
        
        self.play_button.on_click(self._on_play_clicked)
        self.pause_button.on_click(self._on_pause_clicked)

        controls = widgets.HBox([self.play_button, self.pause_button])
        left_panel = widgets.VBox([self.board_output, controls, self.status_label])
        self.ui = widgets.HBox([left_panel, self.move_log])

    def render_board(self, last_move=None):
        """Renders the current chess board using chess.svg inside the Output widget."""
        with self.board_output:
            clear_output(wait=True)
            svg_data = chess.svg.board(
                board=self.board,
                size=self.board_size,
                lastmove=last_move,
                check=self.board.king(self.board.turn) if self.board.is_check() else None
            )
            display(SVG(svg_data))

    def _update_move_log(self):
        """Updates move list in the side log box."""
        moves_san = []
        temp_board = chess.Board()
        for i, move in enumerate(self.board.move_stack):
            if i % 2 == 0:
                moves_san.append(f"{(i//2) + 1}. {temp_board.san(move)}")
            else:
                moves_san[-1] += f"  {temp_board.san(move)}"
            temp_board.push(move)
        
        self.move_log.value = "\n".join(moves_san)

    def display(self):
        """Initializes and displays the UI inside the notebook output cell."""
        display(self.ui)
        self.render_board()

    def _on_play_clicked(self, b):
        if self.board.is_game_over():
            return
        self.is_paused = False
        self.play_button.description = "Resume"
        self.run_game_loop()

    def _on_pause_clicked(self, b):
        self.is_paused = True
        self.status_label.value = "Status: Paused"

    def run_game_loop(self, pgn_filename="ai_vs_ai_match.pgn"):
        """Main non-blocking execution loop driving the AI turns."""
        self.pgn_filename = pgn_filename

        while not self.board.is_game_over() and not self.is_paused:
            # 1. Check for repetition draw conditions before/after moves
            if self.board.can_claim_threefold_repetition():
                self.status_label.value = "Game Over! Draw claimed by 3-fold repetition."
                break

            current_player = self.white_player if self.board.turn == chess.WHITE else self.black_player
            color_str = "White" if self.board.turn == chess.WHITE else "Black"
            player_name = current_player.name if current_player else "Human"

            self.status_label.value = f"Status: {color_str} ({player_name}) thinking..."
            
            # Request move from current AI player
            move = current_player.get_move(self.board.copy())

            if move in self.board.legal_moves:
                self.board.push(move)
                self._update_move_log()
                self.render_board(last_move=move)

                # Check if the move resulted in a repetition draw
                if self.board.can_claim_threefold_repetition() or self.board.is_fivefold_repetition():
                    self.status_label.value = "Game Over! Draw by repetitive moves (Threefold Repetition)."
                    break

                time.sleep(self.turn_delay)
            else:
                self.status_label.value = f"Error: Illegal move by {player_name}"
                break

        # Handle end of game state
        if self.board.is_game_over():
            reason = ""
            if self.board.is_checkmate():
                reason = "Checkmate"
            elif self.board.is_stalemate():
                reason = "Stalemate"
            elif self.board.is_insufficient_material():
                reason = "Insufficient Material"
            elif self.board.can_claim_threefold_repetition() or self.board.is_fivefold_repetition():
                reason = "Threefold Repetition"
            elif self.board.is_fifty_moves():
                reason = "50-Move Rule"
            
            result = self.board.result()
            self.status_label.value = f"Game Over! Result: {result} ({reason})"
            self.export_pgn(self.pgn_filename)
        elif "Draw" in self.status_label.value:
            # Export PGN if ended prematurely by repetition claim
            self.export_pgn(self.pgn_filename)

    def export_pgn(self, filename: str):
        """Exports the recorded game to PGN format."""
        game = chess.pgn.Game.from_board(self.board)
        game.headers["Event"] = "Jupyter Graphical AI Match"
        game.headers["White"] = self.white_player.name if self.white_player else "Human"
        game.headers["Black"] = self.black_player.name if self.black_player else "Human"
        game.headers["Result"] = self.board.result()

        with open(filename, "w") as f:
            exporter = chess.pgn.FileExporter(f)
            game.accept(exporter)


In [46]:
class PGNViewer:
    """
    Reads a PGN file and provides an interactive UI with Back / Next buttons.
    """
    def __init__(self, pgn_filename):
        with open(pgn_filename) as f:
            self.game = chess.pgn.read_game(f)

        self.history = []
        self.moves = []
        
        # Parse all moves and build positions
        temp_board = self.game.board()
        self.history.append(temp_board.copy())
        
        for move in self.game.mainline_moves():
            temp_board.push(move)
            self.history.append(temp_board.copy())
            self.moves.append(move)

        self.current_idx = 0

        # UI Elements
        self.output_area = widgets.Output()
        self.btn_first = widgets.Button(description="|< First", button_style='info')
        self.btn_prev = widgets.Button(description="< Prev", button_style='primary')
        self.btn_next = widgets.Button(description="Next >", button_style='primary')
        self.btn_last = widgets.Button(description="Last >|", button_style='info')

        self.btn_first.on_click(self.go_first)
        self.btn_prev.on_click(self.go_prev)
        self.btn_next.on_click(self.go_next)
        self.btn_last.on_click(self.go_last)

        self.controls = widgets.HBox([self.btn_first, self.btn_prev, self.btn_next, self.btn_last])
        
        display(self.controls)
        display(self.output_area)
        self.render()

    def render(self):
        with self.output_area:
            clear_output(wait=True)
            board = self.history[self.current_idx]
            last_m = self.moves[self.current_idx - 1] if self.current_idx > 0 else None
            
            svg = chess.svg.board(
                board, 
                size=400, 
                lastmove=last_m,
                check=board.king(board.turn) if board.is_check() else None
            )
            display(SVG(svg))
            
            # Text Status
            if self.current_idx == 0:
                print("Start Position")
            else:
                last_played = self.moves[self.current_idx - 1]
                move_num = (self.current_idx + 1) // 2
                color = "White" if self.current_idx % 2 != 0 else "Black"
                print(f"Move {self.current_idx}/{len(self.moves)}: {move_num}. {color} played {last_played.uci()}")
            
            if self.current_idx == len(self.history) - 1:
                print(f"Game Finished! Result: {self.game.headers.get('Result', '*')}")

    def go_first(self, b):
        self.current_idx = 0
        self.render()

    def go_prev(self, b):
        if self.current_idx > 0:
            self.current_idx -= 1
            self.render()

    def go_next(self, b):
        if self.current_idx < len(self.history) - 1:
            self.current_idx += 1
            self.render()

    def go_last(self, b):
        self.current_idx = len(self.history) - 1
        self.render()

def load_and_view_pgn(filename="game.pgn"):
    return PGNViewer(filename)

In [47]:
def ai_selector(class_name, *args, **kwargs):
    # Lookup the class in the current global symbol table
    cls = globals().get(class_name)
    if cls is None:
        raise ValueError(f"Class '{class_name}' not found.")
    return cls(*args, **kwargs)

In [48]:
class MMQSPST:
    """
    Advanced Chess AI featuring:
    - Negamax with Alpha-Beta Pruning
    - Transposition Table (TT) with Zobrist FEN hashing
    - Piece-Square Tables (PST) with Tapered King Evaluation
    - Move Ordering (MVV-LVA / Transposition Move)
    - Quiescence Search for tactile tactical evaluation
    """
    EXACT, LOWERBOUND, UPPERBOUND = 0, 1, 2

    def __init__(self, name="AdvancedEngine", depth=5):
        self.name = name
        self.max_depth = depth
        self.tt = {}  # Instance Transposition Table

        # Material Evaluation Weights
        self.piece_values = {
            chess.PAWN: 100, chess.KNIGHT: 320, chess.BISHOP: 330,
            chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 20000
        }

        # Piece-Square Tables (PST)
        self.pawn_pst = [
             0,  0,  0,  0,  0,  0,  0,  0,
             5, 10, 10,-20,-20, 10, 10,  5,
             5, -5,-10,  0,  0,-10, -5,  5,
            10, 10, 15, 25, 25, 15, 10, 10,
            20, 20, 30, 40, 40, 30, 20, 20,
            40, 40, 50, 60, 60, 50, 40, 40,
            80, 80, 90,100,100, 90, 80, 80,
             0,  0,  0,  0,  0,  0,  0,  0
        ]
        self.knight_pst = [
           -50,-40,-30,-30,-30,-30,-40,-50,
           -40,-20,  0,  5,  5,  0,-20,-40,
           -30,  5, 10, 15, 15, 10,  5,-30,
           -30,  0, 15, 20, 20, 15,  0,-30,
           -30,  5, 15, 20, 20, 15,  5,-30,
           -30,  0, 10, 15, 15, 10,  0,-30,
           -40,-20,  0,  0,  0,  0,-20,-40,
           -50,-40,-30,-30,-30,-30,-40,-50
        ]
        self.bishop_pst = [
           -20,-10,-10,-10,-10,-10,-10,-20,
           -10,  5,  0,  0,  0,  0,  5,-10,
           -10, 10, 10, 10, 10, 10, 10,-10,
           -10,  0, 10, 10, 10, 10,  0,-10,
           -10,  5,  5, 10, 10,  5,  5,-10,
           -10,  0,  5, 10, 10,  5,  0,-10,
           -10,  0,  0,  0,  0,  0,  0,-10,
           -20,-10,-10,-10,-10,-10,-10,-20
        ]
        self.rook_pst = [
             0,  0,  0,  5,  5,  0,  0,  0,
            -5,  0,  0,  0,  0,  0,  0, -5,
            -5,  0,  0,  0,  0,  0,  0, -5,
            -5,  0,  0,  0,  0,  0,  0, -5,
            -5,  0,  0,  0,  0,  0,  0, -5,
            -5,  0,  0,  0,  0,  0,  0, -5,
             5, 10, 10, 10, 10, 10, 10,  5,
             0,  0,  0,  0,  0,  0,  0,  0
        ]
        self.queen_pst = [
           -20,-10,-10, -5, -5,-10,-10,-20,
           -10,  0,  5,  0,  0,  0,  0,-10,
           -10,  5,  5,  5,  5,  5,  0,-10,
            -5,  0,  5,  5,  5,  5,  0, -5,
             0,  0,  5,  5,  5,  5,  0, -5,
           -10,  0,  5,  5,  5,  5,  0,-10,
           -10,  0,  0,  0,  0,  0,  0,-10,
           -20,-10,-10, -5, -5,-10,-10,-20
        ]
        self.king_mg_pst = [
            20, 30, 10,  0,  0, 10, 30, 20,
            20, 20,  0,  0,  0,  0, 20, 20,
           -10,-20,-20,-20,-20,-20,-20,-10,
           -20,-30,-30,-40,-40,-30,-30,-20,
           -30,-40,-40,-50,-50,-40,-40,-30,
           -30,-40,-40,-50,-50,-40,-40,-30,
           -30,-40,-40,-50,-50,-40,-40,-30,
           -30,-40,-40,-50,-50,-40,-40,-30
        ]
        self.king_eg_pst = [
           -50,-40,-30,-20,-20,-30,-40,-50,
           -30,-20,-10,  0,  0,-10,-20,-30,
           -30,-10, 20, 30, 30, 20,-10,-30,
           -30,-10, 30, 40, 40, 30,-10,-30,
           -30,-10, 30, 40, 40, 30,-10,-30,
           -30,-10, 20, 30, 30, 20,-10,-30,
           -30,-30,  0,  0,  0,  0,-30,-30,
           -50,-30,-30,-30,-30,-30,-30,-50
        ]

        self.pst_tables = {
            chess.PAWN: self.pawn_pst,
            chess.KNIGHT: self.knight_pst,
            chess.BISHOP: self.bishop_pst,
            chess.ROOK: self.rook_pst,
            chess.QUEEN: self.queen_pst,
        }

    def evaluate_board(self, board: chess.Board) -> int:
        """Evaluates static score based on material, positional PSTs, and game phase."""
        if board.is_stalemate() or board.is_insufficient_material():
            return 0

        score = 0
        phase = 0

        for square, piece in board.piece_map().items():
            pt = piece.piece_type
            color = piece.color
            
            if pt in (chess.KNIGHT, chess.BISHOP):
                phase += 1
            elif pt == chess.ROOK:
                phase += 2
            elif pt == chess.QUEEN:
                phase += 4

            base_val = self.piece_values[pt]
            sq_idx = square if color == chess.WHITE else chess.square_mirror(square)

            if pt == chess.KING:
                pst_val = int(((self.king_mg_pst[sq_idx] * phase) + (self.king_eg_pst[sq_idx] * (24 - phase))) / 24)
            else:
                pst_val = self.pst_tables[pt][sq_idx]

            if color == chess.WHITE:
                score += base_val + pst_val
            else:
                score -= base_val + pst_val

        return score

    def order_moves(self, board: chess.Board, moves, hash_move=None):
        """Orders moves to prioritize hash move, MVV-LVA captures, and promotions."""
        def score_move(move):
            if move == hash_move:
                return 10000000
                
            score = 0
            if board.is_capture(move):
                if board.is_en_passant(move):
                    return 100005
                
                victim = board.piece_at(move.to_square)
                attacker = board.piece_at(move.from_square)
                if victim and attacker:
                    score = 100000 + self.piece_values[victim.piece_type] - self.piece_values[attacker.piece_type]
            
            if move.promotion:
                score += 90000 + self.piece_values[move.promotion]
                
            return score

        return sorted(moves, key=score_move, reverse=True)

    def quiescence_search(self, board: chess.Board, alpha: int, beta: int, color: int) -> int:
        """Evaluates tactical sequences (captures only) at leaf nodes to avoid horizon effect."""
        stand_pat = self.evaluate_board(board) * color
        if stand_pat >= beta:
            return beta
        if alpha < stand_pat:
            alpha = stand_pat

        captures = list(board.generate_legal_captures())
        for move in self.order_moves(board, captures):
            board.push(move)
            score = -self.quiescence_search(board, -beta, -alpha, -color)
            board.pop()

            if score >= beta:
                return beta
            if score > alpha:
                alpha = score

        return alpha

    def negamax(self, board: chess.Board, depth: int, alpha: int, beta: int, color: int) -> int:
        """Negamax recursive search with TT lookup and Alpha-Beta cutoffs."""
        original_alpha = alpha
        hash_key = board.fen()
        hash_move = None

        if hash_key in self.tt:
            tt_depth, tt_flag, tt_score, tt_move = self.tt[hash_key]
            if tt_depth >= depth:
                if tt_flag == self.EXACT:
                    return tt_score
                if tt_flag == self.LOWERBOUND and tt_score >= beta:
                    return tt_score
                if tt_flag == self.UPPERBOUND and tt_score <= alpha:
                    return tt_score
            hash_move = tt_move

        if depth == 0:
            return self.quiescence_search(board, alpha, beta, color)

        moves = list(board.legal_moves)
        if not moves:
            if board.is_check():
                return -99999 + depth
            return 0

        best_move = None
        best_score = -9999999
        ordered_moves = self.order_moves(board, moves, hash_move)

        for move in ordered_moves:
            board.push(move)
            score = -self.negamax(board, depth - 1, -beta, -alpha, -color)
            board.pop()

            if score > best_score:
                best_score = score
                best_move = move

            alpha = max(alpha, best_score)
            if alpha >= beta:
                break

        if best_score <= original_alpha:
            flag = self.UPPERBOUND
        elif best_score >= beta:
            flag = self.LOWERBOUND
        else:
            flag = self.EXACT
            
        self.tt[hash_key] = (depth, flag, best_score, best_move)

        return best_score

    def get_move(self, board: chess.Board) -> chess.Move:
        """
        Public interface used by GameManager.
        Iterative deepening search up to `self.max_depth`.
        """
        best_move = None
        color = 1 if board.turn == chess.WHITE else -1

        for d in range(1, self.max_depth + 1):
            score = self.negamax(board, d, -9999999, 9999999, color)
            tt_entry = self.tt.get(board.fen())
            if tt_entry:
                best_move = tt_entry[3]
            if abs(score) > 90000:  # Early exit if forced checkmate is detected
                break

        return best_move

In [49]:
class Chungy:
    """
    Standard Minimax + Alpha-Beta Pruning Engine.
    Exposes a clean interface: `get_move(board) -> chess.Move`
    """
    def __init__(self, name="Minimax-Bot", depth=3):
        self.name = name
        self.depth = depth
        self.piece_values = {
            chess.PAWN: 10,
            chess.KNIGHT: 30,
            chess.BISHOP: 30,
            chess.ROOK: 50,
            chess.QUEEN: 90,
            chess.KING: 900
        }

    def evaluate_board(self, board: chess.Board) -> int:
        if board.is_checkmate():
            return -9999 if board.turn == chess.WHITE else 9999
        if board.is_game_over():
            return 0

        score = 0
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                val = self.piece_values[piece.piece_type]
                score += val if piece.color == chess.WHITE else -val
        return score

    def _minimax(self, board: chess.Board, depth: int, alpha: float, beta: float, maximizing: bool):
        if depth == 0 or board.is_game_over():
            return self.evaluate_board(board), None

        best_move = None
        if maximizing:
            max_eval = -float('inf')
            for move in board.legal_moves:
                board.push(move)
                eval_score, _ = self._minimax(board, depth - 1, alpha, beta, False)
                board.pop()
                if eval_score > max_eval:
                    max_eval = eval_score
                    best_move = move
                alpha = max(alpha, eval_score)
                if beta <= alpha:
                    break
            return max_eval, best_move
        else:
            min_eval = float('inf')
            for move in board.legal_moves:
                board.push(move)
                eval_score, _ = self._minimax(board, depth - 1, alpha, beta, True)
                board.pop()
                if eval_score < min_eval:
                    min_eval = eval_score
                    best_move = move
                beta = min(beta, eval_score)
                if beta <= alpha:
                    break
            return min_eval, best_move

    def get_move(self, board: chess.Board) -> chess.Move:
        """Called by GameManager each turn."""
        is_white = (board.turn == chess.WHITE)
        _, move = self._minimax(
            board, 
            depth=self.depth, 
            alpha=-float('inf'), 
            beta=float('inf'), 
            maximizing=is_white
        )
        return move

In [50]:

white_AI = "MMQSPST"
black_AI = "MMQSPST"
white_depth=3
black_depth=3
GAME_NAME = white_AI + str(white_depth) + "_" + black_AI + str(black_depth)

# --- Execution Example ---
if __name__ == "__main__":
    # 1. Instantiate AI engines (using corrected ChessAI class from previous turn)
    ai_white = ai_selector(class_name = white_AI,
                           name="White-Minimax-D3",
                           depth=white_depth)
    ai_black = ai_selector(class_name = black_AI,
                           name="Advanced-Bot-D4",
                           depth=black_depth)

    # 2. Initialize Graphical Manager
    gui_manager = GraphicalGameManager(
        white_player=ai_white,
        black_player=ai_black,
        turn_delay=0.6,
        board_size=380
        ,pgn_filename = GAME_NAME
    )

    # 3. Render graphical UI in the notebook cell
    gui_manager.display()

In [51]:
load_and_view_pgn("arena logs/" + GAME_NAME + ".pgn")

Output()